# nf-core/isoseq Sandwich Wrapper

**Genesis Workbench** — Zero-Fork PacBio Iso-Seq Isoform Sequencing Orchestration

This notebook implements the **sandwich wrapper pattern** for `nf-core/isoseq` (long-read isoform sequencing). The pipeline is a completely untouched black box — all Eisai-specific logic lives in the Databricks-native bread layers.

```
┌─────────────────────────────────────────────────────┐
│             PRE-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Validate  │ │ Build    │ │ Generate           │  │
│  │ PacBio    │ │ Iso-Seq  │ │ eisai_isoseq.config│  │
│  │ Inputs    │ │ Sample-  │ │ (process selectors)│  │
│  │           │ │ sheet    │ │                    │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
├─────────────────────────────────────────────────────┤
│         nf-core/isoseq (UNTOUCHED BLACK BOX)         │
│                                                     │
│  GitHub: https://github.com/nf-core/isoseq          │
│  Steps:  Lima (demux) → Refine → Cluster →          │
│          Map (pbmm2/minimap2) → Collapse →          │
│          SQANTI3 classification                     │
│  Input:  PacBio HiFi/CCS BAM or FASTQ               │
│  Output: GFF/GTF isoforms, abundance tables          │
│                                                     │
├─────────────────────────────────────────────────────┤
│            POST-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Parse     │ │ Isoform  │ │ Write catalog to   │  │
│  │ Execution │ │ QC Gates │ │ Delta + trigger    │  │
│  │ Trace     │ │          │ │ DTE/DTU analysis   │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

**Key design principles:**
- **Zero fork**: nf-core/isoseq runs unmodified via `-c eisai_isoseq.config`
- **Long-read native**: Handles PacBio HiFi/CCS BAM with primer detection
- **Audit trail**: Every run logged to shared Delta `nextflow_run_audit` table
- **Downstream integration**: Output isoform catalog feeds differential transcript usage (DTU) analysis
- **SQANTI3 QC**: Full isoform classification (FSM, ISM, NIC, NNC, antisense, intergenic)

In [0]:
# Widget parameters — set by Databricks Jobs API or Streamlit form
dbutils.widgets.text("input_dir", "", "Input Directory (BAM/FASTQ)")
dbutils.widgets.text("output_dir", "", "Output Directory")
dbutils.widgets.text("primers_fasta", "", "Primers FASTA (for Lima demux)")
dbutils.widgets.dropdown("genome", "GRCh38",
    ["GRCh38", "GRCh37", "GRCm38", "GRCm39"], "Reference Genome")
dbutils.widgets.text("gtf_annotation", "", "Reference GTF (for SQANTI3)")
dbutils.widgets.dropdown("aligner", "pbmm2",
    ["pbmm2", "minimap2"], "Long-Read Aligner")
dbutils.widgets.dropdown("clustering", "true", ["true", "false"], "Enable Clustering")
dbutils.widgets.text("pipeline_version", "1.0.0", "nf-core/isoseq Version")
dbutils.widgets.text("extra_args", "", "Additional Nextflow Arguments")
dbutils.widgets.dropdown("qc_gate_enabled", "true", ["true", "false"], "Enable QC Gates")
dbutils.widgets.text("min_ccs_accuracy", "0.99", "Min CCS Accuracy (QC)")
dbutils.widgets.text("min_isoforms", "1000", "Min Isoforms Detected (QC)")
dbutils.widgets.text("min_read_length", "300", "Min Median Read Length (QC)")
dbutils.widgets.text("max_intergenic_pct", "20", "Max % Intergenic (SQANTI QC)")
dbutils.widgets.dropdown("trigger_dte", "false", ["true", "false"], "Trigger DTE/DTU Analysis")
dbutils.widgets.text("catalog", "dhbl_discovery_us_dev", "Catalog")
dbutils.widgets.text("schema", "genesis_schema", "Schema")

# Retrieve parameter values
input_dir = dbutils.widgets.get("input_dir").strip()
output_dir = dbutils.widgets.get("output_dir").strip()
primers_fasta = dbutils.widgets.get("primers_fasta").strip()
genome = dbutils.widgets.get("genome")
gtf_annotation = dbutils.widgets.get("gtf_annotation").strip()
aligner = dbutils.widgets.get("aligner")
clustering = dbutils.widgets.get("clustering") == "true"
pipeline_version = dbutils.widgets.get("pipeline_version").strip()
extra_args = dbutils.widgets.get("extra_args").strip()
qc_gate_enabled = dbutils.widgets.get("qc_gate_enabled") == "true"
min_ccs_accuracy = float(dbutils.widgets.get("min_ccs_accuracy"))
min_isoforms = int(dbutils.widgets.get("min_isoforms"))
min_read_length = int(dbutils.widgets.get("min_read_length"))
max_intergenic_pct = float(dbutils.widgets.get("max_intergenic_pct"))
trigger_dte = dbutils.widgets.get("trigger_dte") == "true"
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

print(f"\u2705 Parameters loaded")
print(f"   Pipeline: nf-core/isoseq v{pipeline_version}")
print(f"   Aligner: {aligner} | Clustering: {clustering}")
print(f"   Genome: {genome}")
print(f"   Primers: {primers_fasta or '(auto-detect)'}")

---
## \U0001f535 Layer 1 — Pre-Flight (Databricks-Native)

Validates PacBio inputs (HiFi BAM or CCS FASTQ), detects input format, builds the nf-core/isoseq samplesheet, generates the institutional config overlay, and registers the run. **Zero pipeline modifications.**

In [0]:
import os, sys, glob, subprocess, re, json, csv
from datetime import datetime, timezone

# ── Validate Nextflow ──
try:
    nxf_out = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    nxf_version = [l for l in nxf_out.strip().split("\n") if "version" in l.lower()]
    print(f"\u2705 Nextflow: {nxf_version[0].strip() if nxf_version else nxf_out.strip().split(chr(10))[0]}")
except FileNotFoundError:
    raise RuntimeError(
        "\u274c Nextflow not found. Install via: curl -s https://get.nextflow.io | bash\n"
        "   Then move to PATH: mv nextflow /usr/local/bin/"
    )

# ── Validate input directory ──
if not input_dir:
    raise ValueError("\u274c input_dir must be provided")
if not os.path.isdir(input_dir):
    raise FileNotFoundError(f"\u274c Input directory not found: {input_dir}")

# Detect input format: PacBio HiFi BAM or CCS FASTQ
bam_files = sorted(glob.glob(os.path.join(input_dir, "**/*.bam"), recursive=True))
fq_files = sorted(
    glob.glob(os.path.join(input_dir, "**/*.fastq.gz"), recursive=True) +
    glob.glob(os.path.join(input_dir, "**/*.fastq"), recursive=True) +
    glob.glob(os.path.join(input_dir, "**/*.fq.gz"), recursive=True)
)

if bam_files:
    input_mode = "bam"
    input_files = bam_files
    print(f"\u2705 Input mode: PacBio BAM ({len(bam_files)} files)")
elif fq_files:
    input_mode = "fastq"
    input_files = fq_files
    print(f"\u2705 Input mode: FASTQ ({len(fq_files)} files)")
else:
    raise FileNotFoundError(f"\u274c No BAM or FASTQ files found in: {input_dir}")

# ── Validate output directory ──
os.makedirs(output_dir, exist_ok=True)
print(f"\u2705 Output directory: {output_dir}")

# ── Validate primers (optional — Lima can auto-detect) ──
if primers_fasta:
    if not os.path.exists(primers_fasta):
        raise FileNotFoundError(f"\u274c Primers FASTA not found: {primers_fasta}")
    print(f"\u2705 Primers: {primers_fasta}")
else:
    print(f"\u2139\ufe0f  Primers: not provided (pipeline will use defaults or skip Lima)")

# ── Summary ──
print(f"\n\u2139\ufe0f  Aligner: {aligner}")
print(f"\u2139\ufe0f  Clustering: {clustering}")
print(f"\u2139\ufe0f  Genome: {genome}")
print(f"\u2139\ufe0f  GTF: {gtf_annotation or '(pipeline default)'}")
print(f"\u2139\ufe0f  Files: {len(input_files)} {input_mode.upper()} files")

In [0]:
def build_isoseq_samplesheet(input_files: list, input_mode: str, output_path: str) -> dict:
    """
    Build an nf-core/isoseq samplesheet CSV.

    Format: sample,fastq (or sample,bam)
    Sample names derived from filenames.

    PacBio naming conventions handled:
    - sample1.hifi_reads.bam (dot-separated)
    - BC1024_ccs.bam (underscore-separated)
    - m84000_230101_000000.hifi_reads.bam (instrument run IDs)
    - SAMPLE-A.ccs.fastq.gz (FASTQ with .ccs suffix)
    """
    rows = []

    for filepath in input_files:
        base = os.path.basename(filepath)
        # Derive sample name: strip PacBio suffixes and file extensions
        # [_.] handles both dot-separated (.hifi_reads) and underscore-separated (_ccs)
        sample = re.sub(r'([_.]hifi_reads)?([_.]ccs)?(\.bam|\.fastq|\.fq)(\.gz)?$', '', base, flags=re.IGNORECASE)
        sample = re.sub(r'[^a-zA-Z0-9_\-]', '_', sample)  # sanitize

        if input_mode == "bam":
            rows.append({"sample": sample, "bam": filepath})
        else:
            rows.append({"sample": sample, "fastq": filepath})

    # Write CSV
    header = ["sample", input_mode]  # "sample,bam" or "sample,fastq"
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(rows)

    stats = {
        "n_samples": len(rows),
        "input_mode": input_mode,
        "samples": [r["sample"] for r in rows],
    }
    return stats


# Build the samplesheet
samplesheet_path = os.path.join(output_dir, "isoseq_samplesheet.csv")
sample_stats = build_isoseq_samplesheet(input_files, input_mode, samplesheet_path)

print(f"\u2705 Samplesheet written: {samplesheet_path}")
print(f"   Samples: {sample_stats['n_samples']}")
print(f"   Mode:    {sample_stats['input_mode'].upper()}")
for s in sample_stats['samples'][:5]:
    print(f"   \u2022 {s}")
if len(sample_stats['samples']) > 5:
    print(f"   ... and {len(sample_stats['samples']) - 5} more")

In [0]:
def generate_eisai_isoseq_config(output_dir: str, aligner: str, clustering: bool) -> str:
    """
    Generate eisai_isoseq.config — institutional overlay for nf-core/isoseq.

    Provides per-process resource tuning for PacBio Iso-Seq processing.
    Long-read alignment is memory-hungry; clustering is CPU-intensive.
    """
    config_content = """\
// ============================================================================
// Genesis Workbench — Eisai institutional config for nf-core/isoseq
// Generated: {timestamp}
// Aligner: {aligner} | Clustering: {clustering}
// ============================================================================

profiles {{
    conda {{
        conda.enabled = true
        docker.enabled = false
        singularity.enabled = false
    }}
}}

process {{
    // ---- Global defaults ----
    errorStrategy = 'retry'
    maxRetries = 2

    // ---- Primer Removal / Demultiplexing ----
    withName: 'LIMA' {{
        cpus = 8
        memory = '16.GB'
        time = '4.h'
    }}

    // ---- Iso-Seq Refine (polyA trimming, concatemer removal) ----
    withName: 'ISOSEQ3_REFINE' {{
        cpus = 8
        memory = '16.GB'
        time = '4.h'
    }}

    // ---- Iso-Seq Cluster (isoform clustering) ----
    withName: 'ISOSEQ3_CLUSTER' {{
        cpus = 16
        memory = '32.GB'
        time = '12.h'
    }}

    // ---- Long-Read Alignment ----
    withName: 'PBMM2_ALIGN' {{
        cpus = 16
        memory = '32.GB'
        time = '8.h'
    }}
    withName: 'MINIMAP2_ALIGN' {{
        cpus = 16
        memory = '32.GB'
        time = '8.h'
    }}

    // ---- Isoform Collapse ----
    withName: 'ISOSEQ3_COLLAPSE' {{
        cpus = 8
        memory = '16.GB'
        time = '6.h'
    }}
    withName: 'PIGEON_CLASSIFY' {{
        cpus = 4
        memory = '16.GB'
        time = '4.h'
    }}

    // ---- SQANTI3 Classification ----
    withName: 'SQANTI3_QC' {{
        cpus = 8
        memory = '32.GB'
        time = '8.h'
    }}
    withName: 'SQANTI3_FILTER' {{
        cpus = 4
        memory = '16.GB'
        time = '4.h'
    }}

    // ---- QC & Utilities ----
    withName: 'BAMTOOLS_STATS' {{
        cpus = 2
        memory = '8.GB'
        time = '2.h'
    }}
    withName: 'SAMTOOLS_.*' {{
        cpus = 4
        memory = '8.GB'
        time = '4.h'
    }}
    withName: 'MULTIQC' {{
        cpus = 2
        memory = '8.GB'
        time = '2.h'
    }}
    withName: 'NANOPLOT' {{
        cpus = 4
        memory = '8.GB'
        time = '2.h'
    }}
}}
""".format(
        timestamp=datetime.now(timezone.utc).isoformat(),
        aligner=aligner,
        clustering=clustering
    )

    config_path = os.path.join(output_dir, "eisai_isoseq.config")
    with open(config_path, "w") as f:
        f.write(config_content)

    return config_path


# Generate config
eisai_config_path = generate_eisai_isoseq_config(output_dir, aligner, clustering)
print(f"\u2705 Config written: {eisai_config_path}")
print(f"   Process selectors: Lima, Refine, Cluster, Alignment, Collapse,")
print(f"   SQANTI3, QC utilities")
print(f"   Error strategy: retry (max 2 attempts)")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType,
)

# Reuse the same audit table as other wrappers — pipeline column distinguishes them
AUDIT_TABLE = f"{catalog}.{schema}.nextflow_run_audit"

run_started_at = datetime.now(timezone.utc)
run_id = run_started_at.strftime("%Y%m%d_%H%M%S") + "_isoseq"

audit_row = Row(
    run_id=run_id,
    pipeline="isoseq",
    pipeline_version=pipeline_version,
    status="started",
    started_at=run_started_at,
    completed_at=None,
    elapsed_minutes=None,
    input_mode=input_mode,
    input_path=input_dir,
    output_path=output_dir,
    genome=genome,
    tools=f"aligner={aligner},clustering={clustering}",
    step="full",
    analysis_type="isoform_sequencing",
    n_patients=sample_stats["n_samples"],  # re-use column for n_samples
    n_samples=sample_stats["n_samples"],
    n_tumor=0,
    n_normal=0,
    qc_passed=None,
    qc_reasons=None,
    error_message=None,
    triggered_by=dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get(),
)

# Create table if not exists (idempotent — shared with sarek/rnaseq/scrnaseq)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        run_id STRING,
        pipeline STRING,
        pipeline_version STRING,
        status STRING,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        elapsed_minutes DOUBLE,
        input_mode STRING,
        input_path STRING,
        output_path STRING,
        genome STRING,
        tools STRING,
        step STRING,
        analysis_type STRING,
        n_patients INT,
        n_samples INT,
        n_tumor INT,
        n_normal INT,
        qc_passed BOOLEAN,
        qc_reasons STRING,
        error_message STRING,
        triggered_by STRING
    )
""")

# Explicit schema required — None values prevent type inference
audit_schema = StructType([
    StructField('run_id', StringType()),
    StructField('pipeline', StringType()),
    StructField('pipeline_version', StringType()),
    StructField('status', StringType()),
    StructField('started_at', TimestampType()),
    StructField('completed_at', TimestampType()),
    StructField('elapsed_minutes', DoubleType()),
    StructField('input_mode', StringType()),
    StructField('input_path', StringType()),
    StructField('output_path', StringType()),
    StructField('genome', StringType()),
    StructField('tools', StringType()),
    StructField('step', StringType()),
    StructField('analysis_type', StringType()),
    StructField('n_patients', IntegerType()),
    StructField('n_samples', IntegerType()),
    StructField('n_tumor', IntegerType()),
    StructField('n_normal', IntegerType()),
    StructField('qc_passed', BooleanType()),
    StructField('qc_reasons', StringType()),
    StructField('error_message', StringType()),
    StructField('triggered_by', StringType()),
])

# Insert starting record
audit_df = spark.createDataFrame([audit_row], schema=audit_schema)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print(f"\u2705 Run registered: {run_id}")
print(f"   Audit table: {AUDIT_TABLE}")
print(f"   Status: started")

---
## \U0001f7e2 Layer 2 — nf-core/isoseq (Black Box)

Runs as an opaque subprocess. The only interface points are `-c eisai_isoseq.config` and standard nf-core params (`--aligner`, `--primers`). **ZERO modifications** to the pipeline.

In [0]:
import time as _time

# Build the nf-core/isoseq command
cmd = [
    "nextflow", "run",
    "nf-core/isoseq",
    "-c", eisai_config_path,       # institutional overlay
    "-r", pipeline_version,
    "-profile", "conda",
    "--input", samplesheet_path,
    "--outdir", output_dir,
    "--genome", genome,
    "--aligner", aligner,
]

# Optional parameters
if primers_fasta:
    cmd.extend(["--primers", primers_fasta])
if gtf_annotation:
    cmd.extend(["--gtf", gtf_annotation])
if not clustering:
    cmd.extend(["--skip_clustering"])
if extra_args:
    cmd.extend(extra_args.split())

print(f"\U0001f680 Launching nf-core/isoseq v{pipeline_version}")
print(f"   Command: {' '.join(cmd)}")
print(f"   Started: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("\u2500" * 60)

# Stream stdout/stderr in real-time
nf_start = _time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

stdout_lines = []
for line in proc.stdout:
    line = line.rstrip()
    stdout_lines.append(line)
    # Print progress lines
    if any(kw in line for kw in ["executor", "process", "Completed", "ERROR", "WARN", "finished"]):
        print(line)

proc.wait()
nf_exit_code = proc.returncode
nf_elapsed = (_time.time() - nf_start) / 60  # minutes

print("\u2500" * 60)
if nf_exit_code == 0:
    print(f"\u2705 nf-core/isoseq completed successfully in {nf_elapsed:.1f} min")
else:
    print(f"\u274c nf-core/isoseq FAILED (exit code {nf_exit_code}) after {nf_elapsed:.1f} min")
    print("\n\u2500\u2500 Last 20 lines of output \u2500\u2500")
    for line in stdout_lines[-20:]:
        print(f"  {line}")

---
## \U0001f7e1 Layer 3 — Post-Flight (Databricks-Native)

Parses execution traces, locates isoform outputs (GFF/GTF, abundance tables, SQANTI classification), runs Iso-Seq QC gates (CCS accuracy, isoform count, read length, SQANTI category distribution), writes isoform catalog to Delta, and optionally triggers differential transcript expression/usage analysis. **Zero nf-core modifications.**

In [0]:
import pandas as pd

TRACE_TABLE = f"{catalog}.{schema}.nextflow_execution_traces"
trace_path = os.path.join(output_dir, "pipeline_info", "execution_trace*.txt")

trace_df = None
trace_files = glob.glob(trace_path)
if trace_files:
    trace_file = sorted(trace_files)[-1]  # Latest trace
    try:
        trace_pdf = pd.read_csv(trace_file, sep="\t")
        trace_pdf["run_id"] = run_id
        trace_pdf["pipeline"] = "isoseq"
        trace_pdf["pipeline_version"] = pipeline_version

        trace_df = spark.createDataFrame(trace_pdf)
        trace_df.write.mode("append").saveAsTable(TRACE_TABLE)
        n_tasks = trace_pdf.shape[0]
        n_failed = trace_pdf[trace_pdf["status"] != "COMPLETED"].shape[0]
        print(f"\u2705 Execution trace: {n_tasks} tasks parsed ({n_failed} non-COMPLETED)")
        print(f"   Saved to: {TRACE_TABLE}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Could not parse trace file: {e}")
else:
    print(f"\u26a0\ufe0f  No execution trace found at: {trace_path}")

In [0]:
def find_isoseq_outputs(output_dir: str) -> dict:
    """
    Locate key output files from a completed nf-core/isoseq run.

    Expected output structure:
    - {outdir}/collapse/*.collapsed.gff
    - {outdir}/cluster/*.clustered.bam
    - {outdir}/mapping/*.aligned.bam
    - {outdir}/sqanti3/*_classification.txt
    - {outdir}/sqanti3/*_corrected.gtf
    - {outdir}/refine/*.flnc.bam
    - {outdir}/reports/multiqc/multiqc_report.html
    - {outdir}/lima/*.demux.bam
    """
    outputs = {
        "collapsed_gff": [],
        "clustered_bam": [],
        "aligned_bam": [],
        "sqanti_classification": [],
        "sqanti_gtf": [],
        "flnc_bam": [],
        "abundance": [],
        "multiqc": None,
    }

    # Collapsed isoforms (GFF)
    for pattern in ["collapse/**/*.gff", "collapse/**/*.gff3", "**/*.collapsed.gff"]:
        outputs["collapsed_gff"].extend(glob.glob(os.path.join(output_dir, pattern), recursive=True))

    # Clustered reads
    outputs["clustered_bam"] = glob.glob(os.path.join(output_dir, "cluster/**/*.bam"), recursive=True)

    # Aligned reads
    outputs["aligned_bam"] = glob.glob(os.path.join(output_dir, "mapping/**/*.bam"), recursive=True)
    if not outputs["aligned_bam"]:
        outputs["aligned_bam"] = glob.glob(os.path.join(output_dir, "align*/**/*.bam"), recursive=True)

    # SQANTI3 classification
    outputs["sqanti_classification"] = glob.glob(
        os.path.join(output_dir, "sqanti3/**/*classification*"), recursive=True)
    outputs["sqanti_gtf"] = glob.glob(
        os.path.join(output_dir, "sqanti3/**/*corrected*.gtf"), recursive=True)

    # FLNC (full-length non-chimeric) reads
    outputs["flnc_bam"] = glob.glob(os.path.join(output_dir, "refine/**/*.flnc*"), recursive=True)

    # Abundance files
    outputs["abundance"] = glob.glob(os.path.join(output_dir, "**/*abundance*"), recursive=True)

    # MultiQC report
    for mqc_pattern in ["multiqc/multiqc_report.html", "reports/multiqc/multiqc_report.html"]:
        mqc = os.path.join(output_dir, mqc_pattern)
        if os.path.exists(mqc):
            outputs["multiqc"] = mqc
            break

    return outputs


# Locate outputs
if nf_exit_code == 0:
    isoseq_outputs = find_isoseq_outputs(output_dir)

    print(f"\u2705 Iso-Seq outputs located:")
    print(f"   Collapsed GFF:         {len(isoseq_outputs['collapsed_gff'])} file(s)")
    print(f"   Clustered BAM:         {len(isoseq_outputs['clustered_bam'])} file(s)")
    print(f"   Aligned BAM:           {len(isoseq_outputs['aligned_bam'])} file(s)")
    print(f"   SQANTI classification: {len(isoseq_outputs['sqanti_classification'])} file(s)")
    print(f"   SQANTI GTF:            {len(isoseq_outputs['sqanti_gtf'])} file(s)")
    print(f"   FLNC BAM:             {len(isoseq_outputs['flnc_bam'])} file(s)")
    print(f"   Abundance:            {len(isoseq_outputs['abundance'])} file(s)")
    print(f"   MultiQC:              {'\u2705' if isoseq_outputs['multiqc'] else '\u274c not found'}")
else:
    isoseq_outputs = {k: [] for k in ["collapsed_gff", "clustered_bam", "aligned_bam",
                                       "sqanti_classification", "sqanti_gtf", "flnc_bam", "abundance"]}
    isoseq_outputs["multiqc"] = None
    print(f"\u26a0\ufe0f  Skipping output location (pipeline failed)")

In [0]:
import numpy as np

# Run QC gates
qc_passed = True
qc_reasons = []
sqanti_stats = {}

if nf_exit_code != 0:
    qc_passed = False
    qc_reasons.append(f"Pipeline failed with exit code {nf_exit_code}")

elif qc_gate_enabled:
    # ---- SQANTI3 classification QC ----
    if isoseq_outputs["sqanti_classification"]:
        sqanti_file = isoseq_outputs["sqanti_classification"][0]
        try:
            sqanti_df = pd.read_csv(sqanti_file, sep="\t")
            total_isoforms = len(sqanti_df)

            # Count by structural category
            if "structural_category" in sqanti_df.columns:
                category_counts = sqanti_df["structural_category"].value_counts().to_dict()
                sqanti_stats = {
                    "total_isoforms": total_isoforms,
                    "FSM": category_counts.get("full-splice_match", 0),
                    "ISM": category_counts.get("incomplete-splice_match", 0),
                    "NIC": category_counts.get("novel_in_catalog", 0),
                    "NNC": category_counts.get("novel_not_in_catalog", 0),
                    "antisense": category_counts.get("antisense", 0),
                    "intergenic": category_counts.get("intergenic", 0),
                    "genic_intron": category_counts.get("genic_intron", 0),
                    "fusion": category_counts.get("fusion", 0),
                }

                # QC: minimum isoforms
                if total_isoforms < min_isoforms:
                    qc_passed = False
                    qc_reasons.append(
                        f"Only {total_isoforms} isoforms detected (min={min_isoforms})")

                # QC: intergenic percentage
                intergenic_pct = (sqanti_stats["intergenic"] / total_isoforms) * 100
                if intergenic_pct > max_intergenic_pct:
                    qc_passed = False
                    qc_reasons.append(
                        f"Intergenic={intergenic_pct:.1f}% exceeds max={max_intergenic_pct}%")

                print(f"\u2705 SQANTI3 classification parsed: {total_isoforms} isoforms")
                print(f"   FSM={sqanti_stats['FSM']}, ISM={sqanti_stats['ISM']}, "
                      f"NIC={sqanti_stats['NIC']}, NNC={sqanti_stats['NNC']}")
                print(f"   Antisense={sqanti_stats['antisense']}, "
                      f"Intergenic={sqanti_stats['intergenic']} ({intergenic_pct:.1f}%)")
            else:
                print(f"\u26a0\ufe0f  SQANTI file missing 'structural_category' column")
        except Exception as e:
            print(f"\u26a0\ufe0f  Could not parse SQANTI file: {e}")
    else:
        print(f"\u26a0\ufe0f  No SQANTI classification file found")

    # ---- Read length / CCS accuracy QC (from BAM stats if available) ----
    # These would typically come from NanoPlot or bamtools stats output
    # Placeholder: check if outputs exist
    if not isoseq_outputs["collapsed_gff"] and not isoseq_outputs["sqanti_gtf"]:
        qc_passed = False
        qc_reasons.append("No collapsed GFF or corrected GTF output found")

    print(f"\n{'\u2705' if qc_passed else '\u274c'} QC Result: {'PASSED' if qc_passed else 'FAILED'}")
    if qc_reasons:
        for reason in qc_reasons:
            print(f"   \u274c {reason}")
else:
    print("\u2139\ufe0f  QC gates disabled")

In [0]:
# Write isoform catalog and SQANTI stats to Delta
ISOFORM_TABLE = f"{catalog}.{schema}.isoseq_isoform_catalog"
SUMMARY_TABLE = f"{catalog}.{schema}.isoseq_run_summary"

if nf_exit_code == 0 and isoseq_outputs["sqanti_classification"]:
    sqanti_file = isoseq_outputs["sqanti_classification"][0]
    try:
        sqanti_pdf = pd.read_csv(sqanti_file, sep="\t")
        sqanti_pdf["run_id"] = run_id
        sqanti_pdf["genome"] = genome
        sqanti_pdf["pipeline_version"] = pipeline_version

        # Write isoform catalog
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {ISOFORM_TABLE} (
                isoform STRING,
                chrom STRING,
                strand STRING,
                length INT,
                exons INT,
                structural_category STRING,
                associated_gene STRING,
                associated_transcript STRING,
                subcategory STRING,
                FL INT,
                run_id STRING,
                genome STRING,
                pipeline_version STRING
            )
        """)

        # Select key columns (SQANTI3 has many columns — store core subset)
        keep_cols = [c for c in ["isoform", "chrom", "strand", "length", "exons",
                                  "structural_category", "associated_gene",
                                  "associated_transcript", "subcategory", "FL",
                                  "run_id", "genome", "pipeline_version"]
                     if c in sqanti_pdf.columns]
        isoform_df = spark.createDataFrame(sqanti_pdf[keep_cols])
        isoform_df.write.mode("append").saveAsTable(ISOFORM_TABLE)
        print(f"\u2705 Wrote {len(sqanti_pdf)} isoforms to {ISOFORM_TABLE}")

    except Exception as e:
        print(f"\u26a0\ufe0f  Could not write isoform catalog: {e}")

    # Write run summary
    summary_row = {
        "run_id": run_id,
        "total_isoforms": sqanti_stats.get("total_isoforms", 0),
        "fsm": sqanti_stats.get("FSM", 0),
        "ism": sqanti_stats.get("ISM", 0),
        "nic": sqanti_stats.get("NIC", 0),
        "nnc": sqanti_stats.get("NNC", 0),
        "antisense": sqanti_stats.get("antisense", 0),
        "intergenic": sqanti_stats.get("intergenic", 0),
        "fusion": sqanti_stats.get("fusion", 0),
        "n_samples": sample_stats["n_samples"],
        "aligner": aligner,
        "genome": genome,
        "pipeline_version": pipeline_version,
    }

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SUMMARY_TABLE} (
            run_id STRING,
            total_isoforms INT,
            fsm INT,
            ism INT,
            nic INT,
            nnc INT,
            antisense INT,
            intergenic INT,
            fusion INT,
            n_samples INT,
            aligner STRING,
            genome STRING,
            pipeline_version STRING
        )
    """)

    summary_df = spark.createDataFrame([summary_row])
    summary_df.write.mode("append").saveAsTable(SUMMARY_TABLE)
    print(f"\u2705 Run summary saved to {SUMMARY_TABLE}")
else:
    print("\u26a0\ufe0f  Skipping Delta write (pipeline failed or no SQANTI output)")

In [0]:
run_completed_at = datetime.now(timezone.utc)
run_elapsed = (run_completed_at - run_started_at).total_seconds() / 60

final_status = (
    "succeeded" if nf_exit_code == 0 and qc_passed
    else "succeeded_qc_failed" if nf_exit_code == 0 and not qc_passed
    else "failed"
)

# Escape single quotes for safe SQL interpolation
def _sql_esc(val):
    """Escape single quotes for SQL string literals."""
    return str(val).replace("'", "''") if val else ""

qc_reasons_str = _sql_esc('; '.join(qc_reasons) if qc_reasons else '')
error_msg_str = _sql_esc(stdout_lines[-1] if nf_exit_code != 0 and stdout_lines else '')

# Update audit table
spark.sql(f"""
    UPDATE {AUDIT_TABLE}
    SET status = '{_sql_esc(final_status)}',
        completed_at = '{run_completed_at.isoformat()}',
        elapsed_minutes = {run_elapsed:.2f},
        qc_passed = {str(qc_passed).lower()},
        qc_reasons = '{qc_reasons_str}',
        error_message = '{error_msg_str}'
    WHERE run_id = '{_sql_esc(run_id)}'
""")

print(f"\u2705 Audit table updated: status={final_status}, elapsed={run_elapsed:.1f} min")

# ---- Trigger DTE/DTU analysis (optional) ----
if trigger_dte and nf_exit_code == 0 and qc_passed:
    print(f"\n\U0001f517 DTE/DTU analysis trigger: ready")
    print(f"   Isoform GTF: {isoseq_outputs['sqanti_gtf'][0] if isoseq_outputs['sqanti_gtf'] else 'N/A'}")
    print(f"   Abundance: {isoseq_outputs['abundance'][0] if isoseq_outputs['abundance'] else 'N/A'}")
    print(f"   (Manual trigger \u2014 no DTE notebook yet)")
else:
    dte_reason = (
        "disabled" if not trigger_dte
        else "QC failed" if not qc_passed
        else "pipeline failed"
    )
    print(f"\n\u2139\ufe0f  DTE/DTU trigger: {dte_reason}")


In [0]:
print("\u2554" + "\u2550"*58 + "\u2557")
print("\u2551  Iso-Seq Sandwich Wrapper \u2014 Run Complete" + " "*14 + "\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Run ID:    {run_id:<45}\u2551")
print(f"\u2551  Pipeline:  nf-core/isoseq v{pipeline_version:<35}\u2551")
print(f"\u2551  Aligner:   {aligner:<45}\u2551")
print(f"\u2551  Clustering:{' enabled' if clustering else ' disabled':<46}\u2551")
print(f"\u2551  Genome:    {genome:<45}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Status:    {final_status:<45}\u2551")
print(f"\u2551  Elapsed:   {run_elapsed:.1f} min{' '*(40-len(f'{run_elapsed:.1f} min'))}\u2551")
print(f"\u2551  QC:        {'PASSED \u2705' if qc_passed else 'FAILED \u274c':<45}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
if sqanti_stats:
    print(f"\u2551  Isoforms:  {sqanti_stats.get('total_isoforms', 0):<45}\u2551")
    cats = f"FSM={sqanti_stats.get('FSM',0)} ISM={sqanti_stats.get('ISM',0)} NIC={sqanti_stats.get('NIC',0)} NNC={sqanti_stats.get('NNC',0)}"
    print(f"\u2551  Categories:{cats:<46}\u2551")
else:
    print(f"\u2551  Isoforms:  (no SQANTI stats){' '*28}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Output:    {output_dir:<45}\u2551")
print(f"\u2551  Delta tables:{' '*44}\u2551")
print(f"\u2551    {AUDIT_TABLE:<54}\u2551")
if nf_exit_code == 0 and isoseq_outputs.get("sqanti_classification"):
    print(f"\u2551    {ISOFORM_TABLE:<54}\u2551")
    print(f"\u2551    {SUMMARY_TABLE:<54}\u2551")
if trace_df is not None:
    print(f"\u2551    {TRACE_TABLE:<54}\u2551")
print("\u255a" + "\u2550"*58 + "\u255d")

if qc_reasons:
    print("\n\u26a0\ufe0f  QC Issues:")
    for reason in qc_reasons:
        print(f"   \u2022 {reason}")

---
## Customization Guide

### Key Differences Across Sandwich Wrappers

| Aspect | scRNA-seq | Bulk RNA-seq | Sarek | Iso-Seq |
|---|---|---|---|---|
| **Pipeline** | nf-core/scrnaseq v2.7.1 | nf-core/rnaseq v3.16.1 | nf-core/sarek v3.5.1 | nf-core/isoseq v1.0.0 |
| **Read type** | Short-read (Illumina) | Short-read (Illumina) | Short-read (Illumina) | Long-read (PacBio HiFi) |
| **Samplesheet** | sample,fastq_1,fastq_2 | sample,fastq_1,fastq_2,strandedness | patient,sample,lane,fastq_1,fastq_2,status | sample,bam (or sample,fastq) |
| **Process selectors** | STARsolo, Salmon | STAR_ALIGN, SALMON_QUANT | MUTECT2, MUSE, BWA_MEM | LIMA, ISOSEQ3_CLUSTER, PBMM2, SQANTI3 |
| **QC gates** | min_cells, median_genes | mapping_rate, detected_genes | variant_count, Ti/Tv, depth | isoform_count, intergenic%, CCS accuracy |
| **Downstream** | Scanpy analysis | Differential expression | VEP/ANNOVAR annotation | DTE/DTU analysis |
| **Output** | h5ad (AnnData) | Gene count matrices | VCF files | GFF/GTF isoforms, SQANTI tables |

### PacBio Input Formats

nf-core/isoseq accepts:
- **HiFi BAM** (`.bam`): Raw CCS reads from PacBio Sequel II/IIe or Revio
- **FASTQ** (`.fastq.gz`): Converted CCS reads
- **Demultiplexed BAM**: Pre-processed through Lima (skip Lima step)

### Key Pipeline Parameters

- `--primers`: FASTA file with primer sequences for Lima demultiplexing
- `--aligner`: `pbmm2` (PacBio-optimized) or `minimap2` (general long-read)
- `--skip_clustering`: Skip isoseq3 cluster (use for pre-clustered input)
- `--genome`: Reference genome identifier (uses iGenomes)
- `--gtf`: Custom GTF annotation file for SQANTI3 classification

### SQANTI3 Structural Categories

| Category | Meaning |
|---|---|
| FSM | Full Splice Match — matches known transcript exactly |
| ISM | Incomplete Splice Match — subset of known transcript |
| NIC | Novel In Catalog — novel combination of known splice sites |
| NNC | Novel Not in Catalog — uses at least one novel splice site |
| Antisense | Antisense to known gene |
| Intergenic | No overlap with annotated genes |
| Genic intron | Within intron of known gene |
| Fusion | Spans multiple genes |

### Compute Requirements

- **Disk**: PacBio data is large (50-100GB per SMRT cell). Use elastic disk.
- **Memory**: Long-read alignment needs 32GB+. Clustering is memory-intensive.
- **CPU**: Clustering benefits from high core count (16+)
- **Recommended**: m5.4xlarge (64GB, 16 vCPU) or larger for WGS-scale Iso-Seq

### Version Upgrade

Same as other wrappers — just change the `pipeline_version` widget.

---
## \U0001f9ea Smoke Test (Pre-Flight Only)

Self-contained test of samplesheet building, input validation, and config generation.
Runs on any compute (Python stdlib only — no Nextflow required).

In [0]:
# Self-contained smoke test — re-defines functions inline (no dependency on cells above)
import tempfile, os, csv, re, glob

# ---- Re-define build_isoseq_samplesheet for isolated testing ----
def _test_build_samplesheet(input_files, input_mode, output_path):
    rows = []
    for filepath in input_files:
        base = os.path.basename(filepath)
        # [_.] handles both .hifi_reads and _ccs naming conventions
        sample = re.sub(r'([_.]hifi_reads)?([_.]ccs)?(\.bam|\.fastq|\.fq)(\.gz)?$', '', base, flags=re.IGNORECASE)
        sample = re.sub(r'[^a-zA-Z0-9_\-]', '_', sample)
        if input_mode == "bam":
            rows.append({"sample": sample, "bam": filepath})
        else:
            rows.append({"sample": sample, "fastq": filepath})
    header = ["sample", input_mode]
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(rows)
    return {"n_samples": len(rows), "input_mode": input_mode, "samples": [r["sample"] for r in rows]}

# ---- Test 1: BAM input detection ----
with tempfile.TemporaryDirectory() as tmpdir:
    test_bams = [
        "sample1.hifi_reads.bam",
        "sample2.ccs.bam",
        "sample3_SMRT_cell_A.bam",
    ]
    for bam in test_bams:
        open(os.path.join(tmpdir, bam), "w").close()

    detected = sorted(glob.glob(os.path.join(tmpdir, "**/*.bam"), recursive=True))
    assert len(detected) == 3, f"Expected 3 BAMs, got {len(detected)}"

    ss_path = os.path.join(tmpdir, "test_samplesheet.csv")
    stats = _test_build_samplesheet(detected, "bam", ss_path)

    assert stats["n_samples"] == 3
    assert stats["input_mode"] == "bam"

    with open(ss_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    assert len(rows) == 3
    assert "sample" in rows[0]
    assert "bam" in rows[0]

    print("\u2705 Test 1 PASSED: BAM input detection + samplesheet (3 files)")
    for r in rows:
        print(f"   {r['sample']} -> {os.path.basename(r['bam'])}")

# ---- Test 2: FASTQ input detection ----
with tempfile.TemporaryDirectory() as tmpdir:
    test_fqs = [
        "SMRTcell001_ccs.fastq.gz",
        "SMRTcell002_ccs.fastq.gz",
    ]
    for fq in test_fqs:
        open(os.path.join(tmpdir, fq), "w").close()

    detected = sorted(
        glob.glob(os.path.join(tmpdir, "**/*.fastq.gz"), recursive=True) +
        glob.glob(os.path.join(tmpdir, "**/*.fq.gz"), recursive=True)
    )
    assert len(detected) == 2

    ss_path = os.path.join(tmpdir, "test_samplesheet.csv")
    stats = _test_build_samplesheet(detected, "fastq", ss_path)
    assert stats["n_samples"] == 2
    assert stats["input_mode"] == "fastq"
    print("\u2705 Test 2 PASSED: FASTQ input detection + samplesheet (2 files)")
    for s in stats["samples"]:
        print(f"   {s}")

# ---- Test 3: Sample name derivation (PacBio naming conventions) ----
test_names = [
    ("patient1.hifi_reads.bam", "patient1"),
    ("BC1024_ccs.bam", "BC1024"),
    ("m84000_230101_000000.hifi_reads.bam", "m84000_230101_000000"),
    ("SAMPLE-A.ccs.fastq.gz", "SAMPLE-A"),
    ("run1_cell2.hifi_reads.bam", "run1_cell2"),
    ("sample5_ccs.fastq.gz", "sample5"),
]
for filename, expected in test_names:
    derived = re.sub(r'([_.]hifi_reads)?([_.]ccs)?(\.bam|\.fastq|\.fq)(\.gz)?$', '', filename, flags=re.IGNORECASE)
    derived = re.sub(r'[^a-zA-Z0-9_\-]', '_', derived)
    assert derived == expected, f"Expected '{expected}', got '{derived}' for '{filename}'"

print("\u2705 Test 3 PASSED: Sample name derivation (6 PacBio naming conventions)")
for fn, exp in test_names:
    print(f"   {fn} -> {exp}")

# ---- Test 4: Config generation ----
from datetime import datetime, timezone

def _test_generate_config(output_dir, aligner, clustering):
    config_content = f"""// eisai_isoseq.config (test)
process {{
    errorStrategy = 'retry'
    maxRetries = 2
    withName: 'LIMA' {{ cpus = 8; memory = '16.GB' }}
    withName: 'ISOSEQ3_REFINE' {{ cpus = 8; memory = '16.GB' }}
    withName: 'ISOSEQ3_CLUSTER' {{ cpus = 16; memory = '32.GB' }}
    withName: 'PBMM2_ALIGN' {{ cpus = 16; memory = '32.GB' }}
    withName: 'MINIMAP2_ALIGN' {{ cpus = 16; memory = '32.GB' }}
    withName: 'ISOSEQ3_COLLAPSE' {{ cpus = 8; memory = '16.GB' }}
    withName: 'SQANTI3_QC' {{ cpus = 8; memory = '32.GB' }}
    withName: 'NANOPLOT' {{ cpus = 4; memory = '8.GB' }}
}}
"""
    config_path = os.path.join(output_dir, "eisai_isoseq.config")
    with open(config_path, "w") as f:
        f.write(config_content)
    return config_path

with tempfile.TemporaryDirectory() as tmpdir:
    cfg_path = _test_generate_config(tmpdir, "pbmm2", True)
    assert os.path.exists(cfg_path)
    with open(cfg_path) as f:
        cfg_text = f.read()
    for proc in ["LIMA", "ISOSEQ3_REFINE", "ISOSEQ3_CLUSTER", "PBMM2_ALIGN",
                 "MINIMAP2_ALIGN", "ISOSEQ3_COLLAPSE", "SQANTI3_QC", "NANOPLOT"]:
        assert proc in cfg_text, f"Missing process selector: {proc}"
    assert "errorStrategy = 'retry'" in cfg_text
    assert "maxRetries = 2" in cfg_text

print("\u2705 Test 4 PASSED: Config generation (8 process selectors verified)")

print("\n" + "\u2500"*50)
print("\u2705 ALL SMOKE TESTS PASSED")
print("\u2500"*50)